<a href="https://colab.research.google.com/github/23RG10/Rz/blob/main/Horarios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import random
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. LÓGICA DEL HORARIO
# ==========================================
def generar_horario_logica(nombres_personas, estructura_dias_horas):
    datos_horario = []

    # ---------------------------------------------------------
    # NUEVA LÓGICA: Contador para equilibrar los turnos
    # Creamos un diccionario que empieza en 0 para cada persona
    # Ejemplo: {'Juan': 0, 'María': 0, 'Carlos': 0}
    # ---------------------------------------------------------
    conteo_turnos = {persona: 0 for persona in nombres_personas}

    for dia, intervalos in estructura_dias_horas.items():
        for intervalo in intervalos:
            intervalo_limpio = intervalo.strip()
            if not intervalo_limpio:
                continue

            if not nombres_personas:
                persona_asignada = "Sin asignar"
            else:
                # 1. Vemos cuál es la cantidad mínima de turnos que tiene alguien ahora mismo
                min_turnos = min(conteo_turnos.values())

                # 2. Sacamos la lista de personas que tienen esa cantidad mínima
                candidatos = [p for p, turnos in conteo_turnos.items() if turnos == min_turnos]

                # 3. Elegimos al azar entre esos candidatos (para que no siga siempre el mismo orden alfabético)
                persona_asignada = random.choice(candidatos)

                # 4. Le sumamos 1 turno al contador de esa persona
                conteo_turnos[persona_asignada] += 1

            datos_horario.append({
                "Día": dia,
                "Hora": intervalo_limpio,
                "Asignado": persona_asignada
            })

    if not datos_horario:
        return None, None

    df_horario = pd.DataFrame(datos_horario)
    df_pivot = df_horario.pivot(index="Hora", columns="Día", values="Asignado")

    dias_ordenados = [d for d in estructura_dias_horas.keys() if d in df_pivot.columns]
    df_pivot = df_pivot.reindex(columns=dias_ordenados)

    # Devolvemos también el conteo final para que puedas comprobar el balanceo
    return df_pivot

# ==========================================
# 2. INTERFAZ GRÁFICA (VERSIÓN ESTABLE COLAB)
# ==========================================

# --- PASO 1: Datos Generales ---
titulo_paso1 = widgets.HTML("<h3>Paso 1: Datos Generales</h3>")
nota_dias = widgets.HTML("<small><i>* Mantén presionado Ctrl (Windows) o Cmd (Mac) para seleccionar varios días.</i></small>")

txt_personas = widgets.Textarea(
    value='Juan, María, Carlos, Ana',
    placeholder='Escribe los nombres separados por comas',
    description='Personas:',
    layout=widgets.Layout(width='80%', height='60px')
)

select_dias = widgets.SelectMultiple(
    options=['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo'],
    value=['Lunes', 'Martes', 'Miércoles'],
    description='Días:',
    layout=widgets.Layout(width='80%')
)

btn_configurar_horas = widgets.Button(
    description='Continuar a Configurar Horas ➡️',
    button_style='info',
    layout=widgets.Layout(width='40%', margin='15px 0px 10px 0px')
)

# --- Contenedores principales (Cajas) ---
# Usamos VBox para garantizar que Colab no pierda los elementos al renderizar
caja_paso1 = widgets.VBox([titulo_paso1, txt_personas, nota_dias, select_dias, btn_configurar_horas])
caja_paso2 = widgets.VBox() # Se llenará al pulsar el botón 1
caja_resultado = widgets.VBox() # Se llenará al pulsar el botón 2

inputs_horas_dias = {}

# --- PASO 2: Configuración de horas ---
btn_generar = widgets.Button(
    description='✅ Finalizar y Generar Horario',
    button_style='success',
    layout=widgets.Layout(width='40%', margin='20px 0px 0px 0px')
)

def click_configurar_horas(b):
    # Vaciamos datos anteriores por si el usuario cambia de opinión y vuelve a pulsar
    inputs_horas_dias.clear()
    caja_resultado.children = []

    instrucciones_paso2 = widgets.HTML(
        "<hr>"
        "<h3>Paso 2: Configuración de Horas</h3>"
        "<p><i><b>Nota:</b> Los cambios se guardan automáticamente mientras escribes. "
        "Modifica las horas a tu gusto y pulsa el botón verde para terminar.</i></p>"
    )

    cajas_dias = []
    for dia in select_dias.value:
        default_val = "08:00-14:00, 16:00-20:00" if dia != "Sábado" else "09:00-13:00"
        txt_hora_dia = widgets.Text(
            value=default_val,
            description=f'{dia}:',
            layout=widgets.Layout(width='80%')
        )
        inputs_horas_dias[dia] = txt_hora_dia
        cajas_dias.append(txt_hora_dia)

    # Aquí introducimos los elementos en la caja 2, incluyendo el botón verde explícitamente
    caja_paso2.children = [instrucciones_paso2, widgets.VBox(cajas_dias), btn_generar]

btn_configurar_horas.on_click(click_configurar_horas)

# --- PASO 3: Generación del resultado ---
output_tabla = widgets.Output()

def click_generar_horario(b):
    with output_tabla:
        clear_output()

        lista_personas = [p.strip() for p in txt_personas.value.split(',') if p.strip()]

        estructura_dias_horas = {}
        for dia, widget_texto in inputs_horas_dias.items():
            horas = [h.strip() for h in widget_texto.value.split(',') if h.strip()]
            estructura_dias_horas[dia] = horas

        if not lista_personas or not estructura_dias_horas:
            display(widgets.HTML("<p style='color:red;'><b>Error:</b> Faltan personas o días por configurar.</p>"))
            return

        horario_final = generar_horario_logica(lista_personas, estructura_dias_horas)

        if horario_final is not None:
            display(horario_final)
        else:
            display(widgets.HTML("<p style='color:red;'><b>Error:</b> Asegúrate de rellenar los intervalos de hora.</p>"))

    # Añadimos la tabla generada a la caja de resultados
    caja_resultado.children = [widgets.HTML("<hr><h3>🎉 Horario Generado</h3>"), output_tabla]

btn_generar.on_click(click_generar_horario)

# ==========================================
# 3. DIBUJAR INTERFAZ PRINCIPAL
# ==========================================
# Solo necesitamos invocar a las cajas maestras
display(caja_paso1)
display(caja_paso2)
display(caja_resultado)

VBox()

VBox()